# Dubois surface inversion evaluation on UAVSAR C3 data

In this notebook we run the Dubois surface inversion on the UAVSAR C3 product using the provided incidence-angle stream. The radar frequency is derived from the NetCDF metadata.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import shutil
from pathlib import Path

import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

from polsarpro.dev.devtools import parse_psp_parameter_string
from polsarpro.dev.io import read_psp_bin
from polsarpro.dev.metrics import summarize_metrics, visualize_errors
from polsarpro.io import open_netcdf_beam
from polsarpro.physical_inversion import dubois_surface_inversion

# change to your local C-PolSARpro install dir
c_psp_dir = "/home/c_psp/Soft/bin/"
os.environ["PATH"] += os.pathsep + f"{c_psp_dir}/data_process_sngl/"
os.environ["PATH"] += os.pathsep + f"{c_psp_dir}/data_convert/"

input_uavsar_nc = Path("/data/psp/test_files/blackw_20801_22010_006_220507_L090_CX_02_ML5X5.nc")
input_test_dir = Path("/data/psp/blackw_20801_22010_006_220507_L090_CX_02_ML5X5/C3")
output_test_dir = Path("/data/psp/res/dubois_c3/")
output_test_dir.mkdir(parents=True, exist_ok=True)

incidence_angle_file = Path(
    "/data/psp/test_files/blackw_20801_22010_006_220507_L090_CX_02.inc"
)
incidence_angle_c_file = output_test_dir / "dubois_incidence_angle_ml5x5_rad.bin"
file_out = Path("/data/psp/res/test_dubois_surface_inversion_uavsar.nc")

with xr.open_dataset(input_uavsar_nc) as raw_ds:
    center_wavelength_cm = raw_ds["metadata"].attrs["Annotation:CenterWavelength"]

# Dubois expects GHz; UAVSAR metadata stores the center wavelength in cm.
freq_ghz = 29.9792458 / float(center_wavelength_cm)

# Data-driven starting point from the multilooked UAVSAR scene:
# - hv/vv 10th percentile ≈ -12.0 dB
# - hh/vv 25th percentile ≈ -4.5 dB
# This keeps only the lower tails of the cross-pol and co-pol ratios,
# which is a good first pass for suppressing volume and double-bounce
# contributions.
thresh1_db = -5.0
thresh2_db = 0
calibration_coeff = 1.0
calibration_flag = 1
angle_unit = 1  # C-PolSARpro: 0 = degrees, 1 = radians

print(f"Derived radar frequency: {freq_ghz:.6f} GHz")


## Load UAVSAR data and incidence angle raster

In [ ]:
C3 = open_netcdf_beam(input_uavsar_nc)
row_dim, col_dim = C3.dims
# nlines = C3.sizes[row_dim]
# ncols = C3.sizes[col_dim]

# use lines from .ann file
# inc.set_rows                                      (pixels)    = 12344                  ; INC Lines
# inc.set_cols                                      (pixels)    = 11248                  ; INC Samples

nlines = 12344
ncols = 11248

# The incidence-angle file is a little-endian float32 stream in radians.
# Read only the raster-sized prefix and reshape it to the UAVSAR grid.
incidence_angle_values = np.fromfile(
    incidence_angle_file,
    dtype="<f4",
    count=nlines * ncols,
)

# reduce to the same dimensions as multilooked 5x5 image
incidence_angle_values[incidence_angle_values == -10000] = np.nan
incidence_angle = xr.DataArray(
    incidence_angle_values.reshape(nlines, ncols),
    dims=(row_dim, col_dim),
    name="incidence_angle",
).coarsen(y=5, x=5, boundary="trim").mean()

# Write the multilooked incidence angle stream for the C run.
if incidence_angle_c_file.exists():
    incidence_angle_c_file.unlink()
incidence_angle.astype(np.float32).values.tofile(incidence_angle_c_file)


## Apply the surface inversion

In [ ]:
if file_out.exists():
    file_out.unlink()

with ProgressBar():
    out_py = dubois_surface_inversion(
        C3,
        incidence_angle=incidence_angle,
        freq_ghz=freq_ghz,
        thresh1=thresh1_db,
        thresh2=thresh2_db,
        calibration_coeff=calibration_coeff,
    )
    out_py.to_netcdf(file_out)


## Inspect results

In [ ]:
out_py = xr.open_dataset(file_out)
out_py

## Run the C version on the C3 input

In [ ]:
fnr, fnc = C3.sizes[row_dim], C3.sizes[col_dim]

input_str = f"""id: {input_test_dir}
od: {output_test_dir}
iodf: C3
ang: {incidence_angle_c_file}
ofr: 0
ofc: 0
fnr: {fnr}
fnc: {fnc}
fr: {freq_ghz}
un: {angle_unit}
caf: {calibration_flag}
cac: {calibration_coeff}
th1: {thresh1_db}
th2: {thresh2_db}
errf: /tmp"""

result = parse_psp_parameter_string(input_str)
os.system(f"surface_inversion_dubois.exe {result}")

# CPSP does not create another config file in output dir
shutil.copy(input_test_dir / "config.txt", output_test_dir)


## Numerical evaluation

In [ ]:
# open python output (comment if using compute)
out_py = xr.open_dataset(file_out)
# open C-PolSARPro outputs
out_c = {}
out_names = (
    "dubois_er",
    "dubois_mv",
    "dubois_ks",
    "dubois_mask_out",
    "dubois_mask_in",
    "dubois_mask_valid_in_out",
)
for name in out_names:
    file_name = output_test_dir / f"{name}.bin"
    out_c[name] = read_psp_bin(file_name)

df = summarize_metrics(out_py, out_c, short_titles=False, verbose=False)
df


In [ ]:
visualize_errors(out_py=out_py, out_c=out_c, sub_az=1, sub_rg=1)

In [ ]:
from polsarpro.util import pauli_rgb
import matplotlib.pyplot as plt

rgb = pauli_rgb(C3).transpose("y", "x", "band")
plt.figure(figsize=(15,10))
plt.subplot(1,2,1)
plt.imshow(rgb, interpolation="none")
plt.subplot(1,2,2)
plt.imshow(out_py.dubois_mask_valid_in_out, interpolation="none")
plt.tight_layout()